## Table of Contents

1. [Project Overview](#project-overview)
2. [Project Objectives](#project-objectives)
3. [Dataset Description](#dataset-description)
4. [Installing Required Libraries](#installing-required-libraries)
5. [Import Required Libraries](#import-required-libraries)
6. [Part A – Setup, Data Loading, and Exploratory Data Analysis](#part-a)
7. [Part B – Data Preprocessing](#part-b)
8. [Part C – Baseline Decision Tree](#part-c)
9. [Part D – Bagging and Random Forest](#part-d)
10. [Part E – Boosting Models](#part-e)
11. [Part F – Overall Model Comparison](#part-f)
12. [Part G – Feature Importance Analysis](#part-g)
13. [Part H – Hyperparameter Tuning and Cross-Validation](#part-h)
14. [Part I – Reflection and Conclusion](#part-i)
15. [Part J – Kaggle Notebook and Submission Information](#part-j)

<a id="project-overview"></a>

# Project Overview

The purpose of this project is to apply and compare several ensemble machine learning methods to a real-world classification problem. The selected Heart Disease dataset contains patient demographic and clinical indicators that will be used to predict whether a patient has heart disease.

A single Decision Tree classifier will first be trained as a baseline model. Its performance will then be compared with Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost, and LightGBM classifiers.

The models will be evaluated using test accuracy, training time, Out-of-Bag scores where applicable, and feature importance. The analysis will also examine how ensemble learning can improve model robustness, reduce overfitting, and improve generalization compared with a single decision tree.

---

<a id="project-objectives"></a>

## Project Objectives

The primary objectives of this project are to:

1. Load and examine the Heart Disease dataset.
2. Identify the dataset's features, target variable, class balance, and possible data-quality issues.
3. Prepare the data for machine learning.
4. Establish a baseline using a single Decision Tree classifier.
5. Train and evaluate Bagging and Random Forest models.
6. Train and evaluate AdaBoost, Gradient Boosting, XGBoost, and LightGBM models.
7. Compare model accuracy, training time, and Out-of-Bag performance where applicable.
8. Analyze and compare feature importance across ensemble models.
9. Examine how ensemble methods affect overfitting and the bias–variance trade-off.
10. Select the most appropriate model for potential real-world deployment.

<a id="dataset-description"></a>


## Dataset Description

The Heart Disease dataset contains clinical and demographic information about patients. Each row represents one patient record, and each column represents a health-related characteristic.

The target variable is `target`:

- `0` indicates the absence of heart disease.
- `1` indicates the presence of heart disease.

The predictor variables include age, sex, chest-pain type, resting blood pressure, cholesterol level, fasting blood sugar, resting electrocardiogram results, maximum heart rate, exercise-induced angina, ST depression, ST slope, number of major vessels, and thalassemia status.

Because the objective is to predict one of two possible target classes, this is a binary classification problem.

<a id="installing-required-libraries"></a>


# Installing Required Libraries

Most of the libraries used in this project are commonly included with Anaconda or standard Jupyter Notebook installations.

Two external ensemble-learning libraries are also required and installed before the library installation block:

- **XGBoost**
- **LightGBM**

---

<a id="import-required-libraries"></a>


# Import Required Libraries

The following libraries are imported to support data manipulation, visualization, model development, evaluation, and ensemble learning throughout this project.

---

In [ ]:
# ==========================================
# Standard Libraries
# ==========================================
import time
import warnings

# ==========================================
# Data Manipulation
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# Visualization
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Scikit-Learn
# ==========================================
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

# ==========================================
# External Libraries
# ==========================================
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ==========================================
# Notebook Settings
# ==========================================
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

print("="*45)
print("Libraries Loaded Successfully")
print("="*45)
print("NumPy")
print("Pandas")
print("Matplotlib")
print("Seaborn")
print("Scikit-Learn")
print("XGBoost")
print("LightGBM")
print("="*45)
print(f"Random State: {RANDOM_STATE}")
print("="*45)

<a id="part-a"></a>

# Part A – Setup, Data Loading, and Exploratory Data Analysis

---

In this section, the Heart Disease dataset is loaded, verified, and explored to understand its structure, variables, and overall data quality before building any machine learning models.

The exploratory data analysis (EDA) includes:

- Loading and validating the dataset
- Examining the dataset structure
- Reviewing data types
- Computing descriptive statistics
- Identifying missing values
- Detecting duplicate records
- Examining unique values
- Exploring the target variable
- Visualizing feature distributions
- Examining feature relationships

---

## A1. Load the Dataset

To improve portability, the notebook automatically detects whether it is running locally or in the Kaggle environment. The appropriate dataset path is selected automatically, allowing the same notebook to execute without modification in either environment.

---

In [ ]:
# ==========================================================
# A1 - Load the Heart Disease Dataset
# ==========================================================

import os

# Detect whether running locally or on Kaggle
if os.path.exists("/kaggle/input"):
    data_path = "/kaggle/input/heart-disease-dataset/heart.csv"
else:
    data_path = "data/heart.csv"

# Load the dataset
heart_df = pd.read_csv(data_path)

# Display dataset information
print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print("Dataset loaded from:")
print(data_path)

print()
print(f"Rows:    {heart_df.shape[0]:,}")
print(f"Columns: {heart_df.shape[1]}")

print("=" * 60)

# Display the first five records
heart_df.head()

## A2. Dataset Structure and Data Types

The dataset structure is examined to verify the number of observations and variables, review each column's data type, and identify any missing values.

The `info()` method displays the column names, non-null counts, data types, and memory usage.

---

In [ ]:
# ==========================================================
# A2 - Dataset Structure and Data Types
# ==========================================================

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"Rows:    {heart_df.shape[0]:,}")
print(f"Columns: {heart_df.shape[1]}")
print()

heart_df.info()

## A3. Descriptive Statistics

Descriptive statistics provide an overview of the numerical characteristics of each variable. The summary includes the count, mean, standard deviation, minimum, quartiles, and maximum values.

Reviewing these values helps identify unusual ranges, possible outliers, and differences in scale among the features.

---

In [ ]:
# ==========================================================
# A3 - Descriptive Statistics
# ==========================================================

heart_df.describe().T

## A4. Missing Values

Missing values can affect model training and may require imputation or removal. The following analysis calculates the number and percentage of missing values in each column.

---

In [ ]:
# ==========================================================
# A4 - Missing Values
# ==========================================================

missing_summary = pd.DataFrame({
    "Missing Values": heart_df.isnull().sum(),
    "Missing Percentage": (
        heart_df.isnull().sum() / len(heart_df) * 100
    )
})

missing_summary

### Missing-Value Observation

The dataset contains no missing values. Therefore, no imputation or removal of observations is required because of incomplete data.

---

## A5. Duplicate Records

Duplicate observations can cause data leakage if identical patient records appear in both the training and testing sets. They may also cause model performance to appear better than it would be on truly unseen data.

The following code identifies the total number and percentage of duplicate records.

---

In [ ]:
# ==========================================================
# A5 - Duplicate Records
# ==========================================================

duplicate_count = heart_df.duplicated().sum()
duplicate_percentage = duplicate_count / len(heart_df) * 100

print("=" * 60)
print("DUPLICATE RECORD ANALYSIS")
print("=" * 60)
print(f"Total records:        {len(heart_df):,}")
print(f"Duplicate records:    {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")
print("=" * 60)

### Duplicate-Record Observation

The dataset contains duplicate patient records. These records will be removed during data preprocessing before the training and testing sets are created.

Removing duplicates before splitting the data reduces the possibility that identical records will appear in both sets and produce overly optimistic model results.

---

## A6. Unique Values

Examining the number of unique values in each column helps distinguish continuous measurements from binary, ordinal, and categorical variables.

Variables with only a small number of unique values are likely encoded categorical features.

---

In [ ]:
# ==========================================================
# A6 - Unique Values by Feature
# ==========================================================

unique_summary = pd.DataFrame({
    "Unique Values": heart_df.nunique(),
    "Example Values": [
        sorted(heart_df[column].dropna().unique().tolist())[:10]
        for column in heart_df.columns
    ]
})

unique_summary

## A7. Target Variable Distribution

The target variable indicates whether heart disease is absent or present:

- `0` = No heart disease
- `1` = Heart disease present

Examining the class distribution determines whether the dataset is balanced or whether one class is substantially more common than the other.

---

In [ ]:
# ==========================================================
# A7 - Target Variable Distribution
# ==========================================================

target_counts = heart_df["target"].value_counts().sort_index()
target_percentages = (
    heart_df["target"].value_counts(normalize=True).sort_index() * 100
)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

target_summary.index = ["No Heart Disease", "Heart Disease"]
target_summary

In [ ]:
# ==========================================================
# A7 - Target Distribution Visualization
# ==========================================================

ax = sns.countplot(
    data=heart_df,
    x="target"
)

ax.set_title("Distribution of Heart Disease Target Classes")
ax.set_xlabel("Target Class")
ax.set_ylabel("Number of Patients")
ax.set_xticks([0, 1])
ax.set_xticklabels(["No Heart Disease", "Heart Disease"])

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

### Target-Distribution Observation

The two target classes are relatively balanced. This is beneficial because the models will have a similar number of examples from each class during training.

A stratified train-test split will still be used to preserve the class proportions in both subsets.

---

## A8. Feature Distributions

Histograms display the distribution of each dataset variable. These plots help reveal feature ranges, skewness, concentration, and possible unusual values.

Because several variables are categorical values represented by numbers, their histograms appear as separate bars rather than continuous curves.

---

In [ ]:
# ==========================================================
# A8 - Feature Distribution Histograms
# ==========================================================

heart_df.hist(
    figsize=(16, 12),
    bins=20,
    edgecolor="black"
)

plt.suptitle(
    "Distributions of Heart Disease Dataset Variables",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()

## A9. Continuous Feature Distributions by Target

The distributions of selected continuous health measurements are compared between patients with and without heart disease.

The selected variables are:

- Age
- Resting blood pressure
- Cholesterol
- Maximum heart rate
- ST depression

These comparisons may reveal features that help distinguish the two target classes.

---

In [ ]:
# ==========================================================
# A9 - Continuous Features by Target Class
# ==========================================================

continuous_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

for feature in continuous_features:
    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=heart_df,
        x=feature,
        hue="target",
        kde=True,
        multiple="layer",
        element="step"
    )

    plt.title(f"Distribution of {feature} by Target Class")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## A10. Boxplots and Potential Outliers

Boxplots are used to identify the central tendency, spread, and possible outliers in the continuous variables.

Potential outliers are not automatically removed because values that appear unusual may still represent valid medical observations. Any decision to remove observations should be supported by clear evidence that the values are errors.

---

In [ ]:
# ==========================================================
# A10 - Continuous Feature Boxplots
# ==========================================================

for feature in continuous_features:
    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=heart_df,
        x=feature
    )

    plt.title(f"Boxplot of {feature}")
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

### Outlier Observation

Several continuous variables contain observations outside the ranges represented by the boxplot whiskers. These values may reflect genuine differences among patients rather than data-entry errors.

The observations will therefore remain in the dataset. Tree-based ensemble models are generally less sensitive to extreme values than distance-based or linear models.

---

## A11. Correlation Analysis

A correlation matrix measures the strength and direction of the linear relationship between each pair of variables.

Values closer to `1` represent stronger positive relationships, values closer to `-1` represent stronger negative relationships, and values near `0` indicate weak linear relationships.

Correlation does not establish causation, but it can help identify features associated with the target and features that contain similar information.

---

In [ ]:
# ==========================================================
# A11 - Correlation Matrix
# ==========================================================

correlation_matrix = heart_df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5
)

plt.title("Correlation Heatmap of Heart Disease Variables")
plt.tight_layout()
plt.show()

## A12. Correlations with the Target Variable

The following table ranks the variables according to their correlation with the target. Positive values indicate that higher feature values are associated with the presence of heart disease, while negative values indicate an inverse relationship.

The absolute correlation column helps compare relationship strength regardless of direction.

---

In [ ]:
# ==========================================================
# A12 - Feature Correlations with Target
# ==========================================================

target_correlations = (
    correlation_matrix["target"]
    .drop("target")
    .sort_values(ascending=False)
)

target_correlation_table = pd.DataFrame({
    "Correlation with Target": target_correlations,
    "Absolute Correlation": target_correlations.abs()
})

target_correlation_table.sort_values(
    by="Absolute Correlation",
    ascending=False
)

In [ ]:
# ==========================================================
# A12 - Target Correlation Visualization
# ==========================================================

sorted_correlations = target_correlations.sort_values()

plt.figure(figsize=(10, 7))

sorted_correlations.plot(kind="barh")

plt.title("Feature Correlations with Heart Disease Target")
plt.xlabel("Correlation Coefficient")
plt.ylabel("Feature")
plt.axvline(0, linewidth=1)
plt.tight_layout()
plt.show()

## A13. Categorical Features and Heart Disease

Several dataset variables represent categories using numerical codes. Count plots compare each category with the heart-disease target.

These plots help identify categories that may occur more frequently among patients with or without heart disease.

---

In [ ]:
# ==========================================================
# A13 - Categorical Features by Target Class
# ==========================================================

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

for feature in categorical_features:
    plt.figure(figsize=(8, 5))

    sns.countplot(
        data=heart_df,
        x=feature,
        hue="target"
    )

    plt.title(f"{feature} by Heart Disease Target")
    plt.xlabel(feature)
    plt.ylabel("Number of Patients")
    plt.legend(
        title="Target",
        labels=["No Heart Disease", "Heart Disease"]
    )
    plt.tight_layout()
    plt.show()

## A14. Exploratory Data Analysis Summary

The Heart Disease dataset contains 1,025 observations, 13 predictor variables, and one binary target variable. The dataset includes a combination of continuous health measurements and categorical variables represented by numerical codes.

The exploratory analysis produced the following findings:

- The dataset contains no missing values.
- The target classes are relatively balanced.
- Duplicate records are present and should be removed before the data is split.
- The features use different numerical ranges, but the planned tree-based models do not require feature scaling.
- Some continuous variables contain possible outliers, but they appear plausible and will remain in the dataset.
- Several predictor variables show relationships with the target.
- The dataset is suitable for binary classification using decision trees and ensemble-learning methods.

The next section prepares the data for model training and evaluation.

---

<a id="part-b"></a>

# Part B – Data Preprocessing

---

Data preprocessing prepares the Heart Disease dataset for machine learning. This section removes duplicate records, separates the predictor variables from the target, and creates training and testing datasets.

The preprocessing process includes:

- Creating a clean copy of the dataset
- Removing duplicate observations
- Separating features and target
- Creating a stratified training and testing split
- Verifying the final dataset dimensions and class distributions
- Determining whether feature scaling is required

---

## B1. Remove Duplicate Records

Duplicate observations are removed before splitting the data. This prevents identical records from appearing in both the training and testing sets, which could cause data leakage and inflate model performance.

A separate DataFrame named `heart_clean` is created so that the original imported dataset remains unchanged.

---

In [ ]:
# ==========================================================
# B1 - Remove Duplicate Records
# ==========================================================

heart_clean = (
    heart_df
    .drop_duplicates()
    .reset_index(drop=True)
)

records_removed = len(heart_df) - len(heart_clean)

print("=" * 60)
print("DUPLICATE REMOVAL COMPLETE")
print("=" * 60)
print(f"Original records:  {len(heart_df):,}")
print(f"Duplicates removed:{records_removed:,}")
print(f"Remaining records: {len(heart_clean):,}")
print("=" * 60)

## B2. Separate Predictor Features and Target

The predictor matrix `X` contains the patient characteristics used by the models. The target vector `y` contains the heart-disease classification that the models will learn to predict.

The `target` column is removed from `X` and retained separately as `y`.

---

In [ ]:
# ==========================================================
# B2 - Separate Features and Target
# ==========================================================

X = heart_clean.drop(columns="target")
y = heart_clean["target"]

print("=" * 60)
print("FEATURE AND TARGET SEPARATION")
print("=" * 60)
print(f"Predictor matrix shape: {X.shape}")
print(f"Target vector shape:    {y.shape}")
print(f"Number of features:     {X.shape[1]}")
print("=" * 60)

print("\nPredictor features:")
print(X.columns.tolist())

## B3. Create Training and Testing Sets

The dataset is divided into training and testing subsets using an 80/20 split:

- **80% training data** is used to train the models.
- **20% testing data** is reserved for evaluating performance on unseen observations.

The split uses `stratify=y` so that both subsets maintain approximately the same class proportions as the complete dataset. A fixed random state ensures that the same split can be reproduced.

---

In [ ]:
# ==========================================================
# B3 - Stratified Train-Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("=" * 60)
print("TRAIN-TEST SPLIT COMPLETE")
print("=" * 60)
print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training targets:  {y_train.shape}")
print(f"Testing targets:   {y_test.shape}")
print("=" * 60)

## B4. Verify Training and Testing Class Distributions

The class distributions are reviewed after splitting the data to verify that stratification preserved similar proportions of patients with and without heart disease.

---

In [ ]:
# ==========================================================
# B4 - Verify Class Distributions
# ==========================================================

class_distribution = pd.DataFrame({
    "Complete Dataset": y.value_counts(normalize=True).sort_index() * 100,
    "Training Set": y_train.value_counts(normalize=True).sort_index() * 100,
    "Testing Set": y_test.value_counts(normalize=True).sort_index() * 100
})

class_distribution.index = [
    "No Heart Disease",
    "Heart Disease"
]

class_distribution

In [ ]:
# ==========================================================
# B4 - Training and Testing Class Distribution Visualization
# ==========================================================

distribution_plot = class_distribution.T

distribution_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Class Distribution Across Dataset Splits")
plt.xlabel("Dataset")
plt.ylabel("Percentage of Observations")
plt.xticks(rotation=0)
plt.legend(title="Target Class")
plt.tight_layout()
plt.show()

## B5. Feature Scaling Decision

Feature scaling is commonly required for algorithms that depend on distance calculations or gradient-based optimization. Examples include k-nearest neighbors, support-vector machines, and logistic regression.

The models used in this project are Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost, and LightGBM. These tree-based methods divide observations using feature thresholds and do not require all variables to be measured on the same scale.

For that reason, the original feature values will be retained without standardization. This also preserves the interpretability of the original measurements.

---

## B6. Preprocessing Verification

The final preprocessing checks confirm that:

- No missing values remain.
- No duplicate records remain.
- The feature and target datasets contain matching numbers of observations.
- The training and testing sets contain the expected number of records.
- The training and testing sets preserve the target-class proportions.

---

In [ ]:
# ==========================================================
# B6 - Final Preprocessing Verification
# ==========================================================

assert heart_clean.isnull().sum().sum() == 0, \
    "Missing values remain in the cleaned dataset."

assert heart_clean.duplicated().sum() == 0, \
    "Duplicate records remain in the cleaned dataset."

assert len(X) == len(y), \
    "The feature matrix and target vector have different lengths."

assert len(X_train) == len(y_train), \
    "Training features and targets have different lengths."

assert len(X_test) == len(y_test), \
    "Testing features and targets have different lengths."

assert len(X_train) + len(X_test) == len(X), \
    "The train-test split does not contain every observation."

print("=" * 60)
print("PREPROCESSING VERIFICATION PASSED")
print("=" * 60)
print("Missing values:             0")
print("Duplicate records:          0")
print(f"Total cleaned observations: {len(heart_clean):,}")
print(f"Training observations:      {len(X_train):,}")
print(f"Testing observations:       {len(X_test):,}")
print(f"Predictor features:         {X.shape[1]}")
print("=" * 60)

## B7. Data Preprocessing Summary

The duplicate records were removed before splitting the data to reduce the risk of data leakage. The cleaned dataset was separated into 13 predictor features and one binary target variable.

An 80/20 stratified split was used to create the training and testing sets. Stratification preserved the relative proportions of patients with and without heart disease in both subsets.

No missing-value treatment was required, and feature scaling was not applied because every model evaluated in this project is based on decision trees. The prepared training and testing datasets are now ready for baseline model development.

---

The next section establishes a performance baseline using a single Decision Tree classifier.

<a id="part-c"></a>


# Part C – Baseline Decision Tree

---

A single Decision Tree classifier is trained first to establish a baseline for comparison with the ensemble-learning models.

Decision Trees are easy to interpret and can model nonlinear relationships, but an unrestricted tree may memorize the training data and overfit. Comparing training and testing accuracy helps determine whether overfitting is occurring.

This section includes:

- Training the baseline Decision Tree
- Measuring training time
- Comparing training and testing accuracy
- Calculating classification metrics
- Displaying the confusion matrix
- Reviewing the classification report

---

## C1. Initialize the Model Results Collection

A list named `model_results` is created to store the performance measurements for each model. This allows the models to be combined into a comparison table later in the notebook.

The recorded measurements include:

- Training accuracy
- Testing accuracy
- Precision
- Recall
- F1 score
- Training time
- Out-of-Bag score where applicable

---

In [ ]:
# ==========================================================
# C1 - Initialize Model Results Collection
# ==========================================================

model_results = []

print("Model results collection initialized.")

## C2. Train the Baseline Decision Tree

The baseline Decision Tree is trained without restricting its maximum depth. This allows the model to grow until the leaves are sufficiently pure or cannot be split further.

A fixed random state is used to make the result reproducible. Training time is measured so that computational performance can later be compared across models.

---

In [ ]:
# ==========================================================
# C2 - Train Baseline Decision Tree
# ==========================================================

decision_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

start_time = time.perf_counter()

decision_tree.fit(X_train, y_train)

decision_tree_training_time = time.perf_counter() - start_time

print("=" * 60)
print("BASELINE DECISION TREE TRAINED")
print("=" * 60)
print(f"Training time: {decision_tree_training_time:.6f} seconds")
print(f"Tree depth:    {decision_tree.get_depth()}")
print(f"Leaf nodes:    {decision_tree.get_n_leaves()}")
print("=" * 60)

## C3. Generate Decision Tree Predictions

The trained model is used to generate predictions for both the training and testing datasets.

Training predictions help assess whether the model learned the training data too closely, while testing predictions measure performance on unseen observations.

---

In [ ]:
# ==========================================================
# C3 - Generate Decision Tree Predictions
# ==========================================================

decision_tree_train_predictions = decision_tree.predict(X_train)
decision_tree_test_predictions = decision_tree.predict(X_test)

print("Training and testing predictions generated successfully.")

## C4. Evaluate the Baseline Decision Tree

The baseline model is evaluated using several classification measurements:

- **Accuracy:** Percentage of all predictions that were correct
- **Precision:** Percentage of predicted positive cases that were actually positive
- **Recall:** Percentage of actual positive cases correctly identified
- **F1 score:** Harmonic mean of precision and recall

Training and testing accuracy are compared to identify possible overfitting.

---

In [ ]:
# ==========================================================
# C4 - Evaluate Baseline Decision Tree
# ==========================================================

decision_tree_train_accuracy = accuracy_score(
    y_train,
    decision_tree_train_predictions
)

decision_tree_test_accuracy = accuracy_score(
    y_test,
    decision_tree_test_predictions
)

decision_tree_precision = precision_score(
    y_test,
    decision_tree_test_predictions,
    zero_division=0
)

decision_tree_recall = recall_score(
    y_test,
    decision_tree_test_predictions,
    zero_division=0
)

decision_tree_f1 = f1_score(
    y_test,
    decision_tree_test_predictions,
    zero_division=0
)

decision_tree_accuracy_gap = (
    decision_tree_train_accuracy - decision_tree_test_accuracy
)

print("=" * 60)
print("BASELINE DECISION TREE PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {decision_tree_train_accuracy:.4f}")
print(f"Testing accuracy:  {decision_tree_test_accuracy:.4f}")
print(f"Accuracy gap:      {decision_tree_accuracy_gap:.4f}")
print(f"Precision:         {decision_tree_precision:.4f}")
print(f"Recall:            {decision_tree_recall:.4f}")
print(f"F1 score:          {decision_tree_f1:.4f}")
print(f"Training time:     {decision_tree_training_time:.6f} seconds")
print("=" * 60)

## C5. Store the Decision Tree Results

The Decision Tree measurements are added to the shared model-results collection. An Out-of-Bag score is not available for a single Decision Tree, so that field is recorded as unavailable.

---

In [ ]:
# ==========================================================
# C5 - Store Decision Tree Results
# ==========================================================

model_results.append({
    "Model": "Decision Tree",
    "Training Accuracy": decision_tree_train_accuracy,
    "Testing Accuracy": decision_tree_test_accuracy,
    "Accuracy Gap": decision_tree_accuracy_gap,
    "Precision": decision_tree_precision,
    "Recall": decision_tree_recall,
    "F1 Score": decision_tree_f1,
    "Training Time": decision_tree_training_time,
    "OOB Score": np.nan
})

print("Decision Tree results stored successfully.")

## C6. Decision Tree Confusion Matrix

The confusion matrix compares the model's predicted classes with the actual classes.

It contains four outcomes:

- **True negatives:** Correctly predicted absence of heart disease
- **False positives:** Heart disease predicted when it was absent
- **False negatives:** Heart disease missed by the model
- **True positives:** Correctly predicted presence of heart disease

In a medical classification setting, false negatives are especially important because they represent patients whose heart disease was not identified by the model.

---

In [ ]:
# ==========================================================
# C6 - Decision Tree Confusion Matrix
# ==========================================================

decision_tree_confusion_matrix = confusion_matrix(
    y_test,
    decision_tree_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=decision_tree_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("Baseline Decision Tree Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

## C7. Decision Tree Classification Report

The classification report presents precision, recall, F1 score, and support for each target class.

Support represents the number of actual observations belonging to each class in the testing dataset.

---

In [ ]:
# ==========================================================
# C7 - Decision Tree Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        decision_tree_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## C8. Decision Tree Overfitting Assessment

The following code interprets the difference between training and testing accuracy.

A large positive gap suggests that the model performs substantially better on the data it was trained on than on unseen data. This is evidence of possible overfitting.

---

In [ ]:
# ==========================================================
# C8 - Decision Tree Overfitting Assessment
# ==========================================================

if decision_tree_accuracy_gap >= 0.10:
    overfitting_assessment = (
        "The Decision Tree shows strong evidence of overfitting."
    )
elif decision_tree_accuracy_gap >= 0.05:
    overfitting_assessment = (
        "The Decision Tree shows moderate evidence of overfitting."
    )
else:
    overfitting_assessment = (
        "The Decision Tree shows limited evidence of overfitting."
    )

print("=" * 60)
print("OVERFITTING ASSESSMENT")
print("=" * 60)
print(overfitting_assessment)
print(f"Training-testing accuracy gap: {decision_tree_accuracy_gap:.4f}")
print("=" * 60)

### Baseline Decision Tree Observation

The baseline Decision Tree provides an initial measure of classification performance. Its training accuracy should be compared carefully with its testing accuracy.

A substantial difference between these values would indicate that the unrestricted tree learned details specific to the training dataset and did not generalize equally well to unseen observations. The ensemble models in the next section are intended to reduce this instability by combining predictions from multiple trees.

---

<a id="part-d"></a>


# Part D – Bagging and Random Forest

---

Bagging and Random Forest are ensemble methods that combine predictions from multiple Decision Trees.

**Bagging**, or bootstrap aggregation, trains multiple trees using different bootstrap samples from the training dataset. Their predictions are combined through majority voting. This process can reduce variance and improve stability compared with a single tree.

**Random Forest** extends Bagging by also selecting a random subset of features at each split. This makes the individual trees less correlated and can further improve generalization.

This section includes:

- Training a Bagging classifier
- Evaluating its test and Out-of-Bag performance
- Training a Random Forest classifier
- Evaluating its test and Out-of-Bag performance
- Comparing both ensemble models with the baseline Decision Tree

---

## D1. Train the Bagging Classifier

The Bagging classifier uses 100 Decision Trees. Each tree is trained on a bootstrap sample created by randomly selecting training observations with replacement.

The Out-of-Bag option is enabled. Observations excluded from an individual tree's bootstrap sample can be used to estimate model performance without using the testing dataset.

---

In [ ]:
# ==========================================================
# D1 - Train Bagging Classifier
# ==========================================================

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.perf_counter()

bagging_model.fit(X_train, y_train)

bagging_training_time = time.perf_counter() - start_time

print("=" * 60)
print("BAGGING CLASSIFIER TRAINED")
print("=" * 60)
print(f"Number of estimators: {bagging_model.n_estimators}")
print(f"Training time:        {bagging_training_time:.6f} seconds")
print(f"Out-of-Bag score:     {bagging_model.oob_score_:.4f}")
print("=" * 60)

In [ ]:
# ==========================================================
# D1 - Train Bagging Classifier
# ==========================================================

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.perf_counter()

bagging_model.fit(X_train, y_train)

bagging_training_time = time.perf_counter() - start_time

print("=" * 60)
print("BAGGING CLASSIFIER TRAINED")
print("=" * 60)
print(f"Number of estimators: {bagging_model.n_estimators}")
print(f"Training time:        {bagging_training_time:.6f} seconds")
print(f"Out-of-Bag score:     {bagging_model.oob_score_:.4f}")
print("=" * 60)

## D2. Evaluate the Bagging Classifier

Predictions are generated for both the training and testing datasets. The same measurements used for the baseline model are calculated so that the results can be compared consistently.

---

In [ ]:
# ==========================================================
# D2 - Evaluate Bagging Classifier
# ==========================================================

bagging_train_predictions = bagging_model.predict(X_train)
bagging_test_predictions = bagging_model.predict(X_test)

bagging_train_accuracy = accuracy_score(
    y_train,
    bagging_train_predictions
)

bagging_test_accuracy = accuracy_score(
    y_test,
    bagging_test_predictions
)

bagging_precision = precision_score(
    y_test,
    bagging_test_predictions,
    zero_division=0
)

bagging_recall = recall_score(
    y_test,
    bagging_test_predictions,
    zero_division=0
)

bagging_f1 = f1_score(
    y_test,
    bagging_test_predictions,
    zero_division=0
)

bagging_accuracy_gap = (
    bagging_train_accuracy - bagging_test_accuracy
)

print("=" * 60)
print("BAGGING CLASSIFIER PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {bagging_train_accuracy:.4f}")
print(f"Testing accuracy:  {bagging_test_accuracy:.4f}")
print(f"Accuracy gap:      {bagging_accuracy_gap:.4f}")
print(f"Precision:         {bagging_precision:.4f}")
print(f"Recall:            {bagging_recall:.4f}")
print(f"F1 score:          {bagging_f1:.4f}")
print(f"Out-of-Bag score:  {bagging_model.oob_score_:.4f}")
print(f"Training time:     {bagging_training_time:.6f} seconds")
print("=" * 60)

In [ ]:
# ==========================================================
# D3 - Store Bagging Results
# ==========================================================

model_results.append({
    "Model": "Bagging",
    "Training Accuracy": bagging_train_accuracy,
    "Testing Accuracy": bagging_test_accuracy,
    "Accuracy Gap": bagging_accuracy_gap,
    "Precision": bagging_precision,
    "Recall": bagging_recall,
    "F1 Score": bagging_f1,
    "Training Time": bagging_training_time,
    "OOB Score": bagging_model.oob_score_
})

print("Bagging results stored successfully.")

## D4. Bagging Confusion Matrix

The Bagging confusion matrix shows how combining multiple bootstrap-trained trees affects correct classifications and classification errors.

---

In [ ]:
# ==========================================================
# D4 - Bagging Confusion Matrix
# ==========================================================

bagging_confusion_matrix = confusion_matrix(
    y_test,
    bagging_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=bagging_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("Bagging Classifier Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# D5 - Bagging Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        bagging_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## D6. Train the Random Forest Classifier

The Random Forest classifier uses 100 Decision Trees trained on bootstrap samples. Unlike standard Bagging, each split considers only a randomly selected subset of predictor features.

This added randomness reduces similarity among the trees and can improve the ensemble's ability to generalize.

Out-of-Bag scoring is enabled to provide an additional estimate of performance.

---

In [ ]:
# ==========================================================
# D6 - Train Random Forest Classifier
# ==========================================================

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.perf_counter()

random_forest_model.fit(X_train, y_train)

random_forest_training_time = time.perf_counter() - start_time

print("=" * 60)
print("RANDOM FOREST CLASSIFIER TRAINED")
print("=" * 60)
print(f"Number of estimators: {random_forest_model.n_estimators}")
print(f"Maximum features:     {random_forest_model.max_features}")
print(f"Training time:        {random_forest_training_time:.6f} seconds")
print(f"Out-of-Bag score:     {random_forest_model.oob_score_:.4f}")
print("=" * 60)

## D7. Evaluate the Random Forest Classifier

The Random Forest is evaluated using the same measurements as the Decision Tree and Bagging models.

Its training-testing accuracy gap and Out-of-Bag score help assess whether the model generalizes more effectively than the baseline tree.

---

In [ ]:
# ==========================================================
# D7 - Evaluate Random Forest Classifier
# ==========================================================

random_forest_train_predictions = random_forest_model.predict(X_train)
random_forest_test_predictions = random_forest_model.predict(X_test)

random_forest_train_accuracy = accuracy_score(
    y_train,
    random_forest_train_predictions
)

random_forest_test_accuracy = accuracy_score(
    y_test,
    random_forest_test_predictions
)

random_forest_precision = precision_score(
    y_test,
    random_forest_test_predictions,
    zero_division=0
)

random_forest_recall = recall_score(
    y_test,
    random_forest_test_predictions,
    zero_division=0
)

random_forest_f1 = f1_score(
    y_test,
    random_forest_test_predictions,
    zero_division=0
)

random_forest_accuracy_gap = (
    random_forest_train_accuracy - random_forest_test_accuracy
)

print("=" * 60)
print("RANDOM FOREST PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {random_forest_train_accuracy:.4f}")
print(f"Testing accuracy:  {random_forest_test_accuracy:.4f}")
print(f"Accuracy gap:      {random_forest_accuracy_gap:.4f}")
print(f"Precision:         {random_forest_precision:.4f}")
print(f"Recall:            {random_forest_recall:.4f}")
print(f"F1 score:          {random_forest_f1:.4f}")
print(f"Out-of-Bag score:  {random_forest_model.oob_score_:.4f}")
print(f"Training time:     {random_forest_training_time:.6f} seconds")
print("=" * 60)

In [ ]:
# ==========================================================
# D8 - Store Random Forest Results
# ==========================================================

model_results.append({
    "Model": "Random Forest",
    "Training Accuracy": random_forest_train_accuracy,
    "Testing Accuracy": random_forest_test_accuracy,
    "Accuracy Gap": random_forest_accuracy_gap,
    "Precision": random_forest_precision,
    "Recall": random_forest_recall,
    "F1 Score": random_forest_f1,
    "Training Time": random_forest_training_time,
    "OOB Score": random_forest_model.oob_score_
})

print("Random Forest results stored successfully.")

In [ ]:
# ==========================================================
# D9 - Random Forest Confusion Matrix
# ==========================================================

random_forest_confusion_matrix = confusion_matrix(
    y_test,
    random_forest_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=random_forest_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("Random Forest Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# D10 - Random Forest Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        random_forest_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## D11. Initial Model Comparison

The first three models are compared using training accuracy, testing accuracy, accuracy gap, precision, recall, F1 score, training time, and Out-of-Bag score.

The accuracy gap helps assess possible overfitting. Smaller gaps generally indicate that training and testing performance are more consistent.

Out-of-Bag scores are available only for the Bagging and Random Forest models because those models use bootstrap sampling.

---

In [ ]:
# ==========================================================
# D11 - Initial Model Comparison Table
# ==========================================================

initial_comparison = pd.DataFrame(model_results)

initial_comparison = initial_comparison.sort_values(
    by="Testing Accuracy",
    ascending=False
).reset_index(drop=True)

initial_comparison.insert(
    0,
    "Rank",
    range(1, len(initial_comparison) + 1)
)

initial_comparison.round(4)

In [ ]:
# ==========================================================
# D12 - Testing Accuracy Comparison
# ==========================================================

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=initial_comparison,
    x="Model",
    y="Testing Accuracy"
)

plt.title("Initial Model Testing Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Testing Accuracy")
plt.ylim(0, 1.05)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# D13 - Training versus Testing Accuracy
# ==========================================================

accuracy_comparison = initial_comparison[
    [
        "Model",
        "Training Accuracy",
        "Testing Accuracy"
    ]
].set_index("Model")

accuracy_comparison.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Training and Testing Accuracy by Model")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()

## D14. Out-of-Bag Score Comparison

The Out-of-Bag score estimates performance using training observations that were not selected for an individual tree's bootstrap sample.

A close relationship between the Out-of-Bag score and testing accuracy suggests that the Out-of-Bag estimate provides a reasonable indication of performance on unseen data.

---

In [ ]:
# ==========================================================
# D14 - Out-of-Bag Score Comparison
# ==========================================================

oob_comparison = initial_comparison[
    initial_comparison["OOB Score"].notna()
][
    [
        "Model",
        "Testing Accuracy",
        "OOB Score"
    ]
].set_index("Model")

oob_comparison.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Testing Accuracy versus Out-of-Bag Score")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# D15 - Training Time Comparison
# ==========================================================

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=initial_comparison,
    x="Model",
    y="Training Time"
)

plt.title("Initial Model Training Time Comparison")
plt.xlabel("Model")
plt.ylabel("Training Time in Seconds")

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3)

plt.tight_layout()
plt.show()

## D16. Bagging and Random Forest Summary

The baseline Decision Tree, Bagging classifier, and Random Forest classifier were evaluated using the same training and testing datasets.

The single Decision Tree provides a useful baseline but may show a noticeable difference between training and testing performance. Bagging reduces variance by combining trees trained on different bootstrap samples. Random Forest adds random feature selection, which reduces correlation among the individual trees.

The comparison should focus on several factors rather than accuracy alone:

- Testing accuracy indicates overall performance on unseen observations.
- Precision measures the reliability of positive predictions.
- Recall measures the model's ability to identify patients with heart disease.
- F1 score balances precision and recall.
- The training-testing accuracy gap provides evidence of possible overfitting.
- The Out-of-Bag score provides an additional estimate of generalization.
- Training time shows the computational cost of each approach.

The ensemble models are expected to provide more stable predictions than a single unrestricted Decision Tree, although the final conclusion will depend on the actual results generated by the notebook.

---

The next section evaluates AdaBoost, Gradient Boosting, XGBoost, and LightGBM.

<a id="part-e"></a>


# Part E – Boosting Models

---

Boosting is an ensemble-learning strategy that builds models sequentially. Each new model focuses on improving the errors made by the models trained before it.

This section evaluates four boosting methods:

- AdaBoost
- Gradient Boosting
- XGBoost
- LightGBM

Each model is trained using the same training data and evaluated using the same testing data. Their performance is measured using accuracy, precision, recall, F1 score, and training time.

---

## E1. AdaBoost Classifier

AdaBoost, or Adaptive Boosting, trains a sequence of weak learners. Each new learner places greater emphasis on observations that were previously misclassified.

The final prediction is produced by combining the weighted predictions of all learners.

---

In [ ]:
# ==========================================================
# E1 - Train and Evaluate AdaBoost
# ==========================================================

adaboost_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=RANDOM_STATE
)

start_time = time.perf_counter()

adaboost_model.fit(X_train, y_train)

adaboost_training_time = time.perf_counter() - start_time

adaboost_train_predictions = adaboost_model.predict(X_train)
adaboost_test_predictions = adaboost_model.predict(X_test)

adaboost_train_accuracy = accuracy_score(
    y_train,
    adaboost_train_predictions
)

adaboost_test_accuracy = accuracy_score(
    y_test,
    adaboost_test_predictions
)

adaboost_precision = precision_score(
    y_test,
    adaboost_test_predictions,
    zero_division=0
)

adaboost_recall = recall_score(
    y_test,
    adaboost_test_predictions,
    zero_division=0
)

adaboost_f1 = f1_score(
    y_test,
    adaboost_test_predictions,
    zero_division=0
)

adaboost_accuracy_gap = (
    adaboost_train_accuracy - adaboost_test_accuracy
)

print("=" * 60)
print("ADABOOST PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {adaboost_train_accuracy:.4f}")
print(f"Testing accuracy:  {adaboost_test_accuracy:.4f}")
print(f"Accuracy gap:      {adaboost_accuracy_gap:.4f}")
print(f"Precision:         {adaboost_precision:.4f}")
print(f"Recall:            {adaboost_recall:.4f}")
print(f"F1 score:          {adaboost_f1:.4f}")
print(f"Training time:     {adaboost_training_time:.6f} seconds")
print("=" * 60)

In [ ]:
# ==========================================================
# E1 - Store AdaBoost Results
# ==========================================================

model_results.append({
    "Model": "AdaBoost",
    "Training Accuracy": adaboost_train_accuracy,
    "Testing Accuracy": adaboost_test_accuracy,
    "Accuracy Gap": adaboost_accuracy_gap,
    "Precision": adaboost_precision,
    "Recall": adaboost_recall,
    "F1 Score": adaboost_f1,
    "Training Time": adaboost_training_time,
    "OOB Score": np.nan
})

print("AdaBoost results stored successfully.")

## E2. AdaBoost Confusion Matrix and Classification Report

The confusion matrix shows the number of correct and incorrect predictions for each target class. The classification report provides class-specific precision, recall, F1 score, and support.

---

In [ ]:
# ==========================================================
# E2 - AdaBoost Confusion Matrix
# ==========================================================

adaboost_confusion_matrix = confusion_matrix(
    y_test,
    adaboost_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=adaboost_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("AdaBoost Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E2 - AdaBoost Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        adaboost_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## E3. Gradient Boosting Classifier

Gradient Boosting trains a sequence of shallow Decision Trees. Each new tree attempts to reduce the remaining prediction error of the combined model.

The learning rate controls how strongly each new tree contributes to the ensemble.

---

In [ ]:
# ==========================================================
# E3 - Train and Evaluate Gradient Boosting
# ==========================================================

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.10,
    max_depth=3,
    random_state=RANDOM_STATE
)

start_time = time.perf_counter()

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_training_time = (
    time.perf_counter() - start_time
)

gradient_boosting_train_predictions = (
    gradient_boosting_model.predict(X_train)
)

gradient_boosting_test_predictions = (
    gradient_boosting_model.predict(X_test)
)

gradient_boosting_train_accuracy = accuracy_score(
    y_train,
    gradient_boosting_train_predictions
)

gradient_boosting_test_accuracy = accuracy_score(
    y_test,
    gradient_boosting_test_predictions
)

gradient_boosting_precision = precision_score(
    y_test,
    gradient_boosting_test_predictions,
    zero_division=0
)

gradient_boosting_recall = recall_score(
    y_test,
    gradient_boosting_test_predictions,
    zero_division=0
)

gradient_boosting_f1 = f1_score(
    y_test,
    gradient_boosting_test_predictions,
    zero_division=0
)

gradient_boosting_accuracy_gap = (
    gradient_boosting_train_accuracy
    - gradient_boosting_test_accuracy
)

print("=" * 60)
print("GRADIENT BOOSTING PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {gradient_boosting_train_accuracy:.4f}")
print(f"Testing accuracy:  {gradient_boosting_test_accuracy:.4f}")
print(f"Accuracy gap:      {gradient_boosting_accuracy_gap:.4f}")
print(f"Precision:         {gradient_boosting_precision:.4f}")
print(f"Recall:            {gradient_boosting_recall:.4f}")
print(f"F1 score:          {gradient_boosting_f1:.4f}")
print(
    f"Training time:     "
    f"{gradient_boosting_training_time:.6f} seconds"
)
print("=" * 60)

In [ ]:
# ==========================================================
# E3 - Store Gradient Boosting Results
# ==========================================================

model_results.append({
    "Model": "Gradient Boosting",
    "Training Accuracy": gradient_boosting_train_accuracy,
    "Testing Accuracy": gradient_boosting_test_accuracy,
    "Accuracy Gap": gradient_boosting_accuracy_gap,
    "Precision": gradient_boosting_precision,
    "Recall": gradient_boosting_recall,
    "F1 Score": gradient_boosting_f1,
    "Training Time": gradient_boosting_training_time,
    "OOB Score": np.nan
})

print("Gradient Boosting results stored successfully.")

In [ ]:
# ==========================================================
# E4 - Gradient Boosting Confusion Matrix
# ==========================================================

gradient_boosting_confusion_matrix = confusion_matrix(
    y_test,
    gradient_boosting_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=gradient_boosting_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("Gradient Boosting Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E4 - Gradient Boosting Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        gradient_boosting_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## E5. XGBoost Classifier

XGBoost is an optimized gradient-boosting implementation designed for speed, regularization, and predictive performance.

It includes controls that reduce overfitting and efficiently handles repeated tree construction.

---

In [ ]:
# ==========================================================
# E5 - Train and Evaluate XGBoost
# ==========================================================

xgboost_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.10,
    max_depth=3,
    subsample=0.80,
    colsample_bytree=0.80,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.perf_counter()

xgboost_model.fit(X_train, y_train)

xgboost_training_time = time.perf_counter() - start_time

xgboost_train_predictions = xgboost_model.predict(X_train)
xgboost_test_predictions = xgboost_model.predict(X_test)

xgboost_train_accuracy = accuracy_score(
    y_train,
    xgboost_train_predictions
)

xgboost_test_accuracy = accuracy_score(
    y_test,
    xgboost_test_predictions
)

xgboost_precision = precision_score(
    y_test,
    xgboost_test_predictions,
    zero_division=0
)

xgboost_recall = recall_score(
    y_test,
    xgboost_test_predictions,
    zero_division=0
)

xgboost_f1 = f1_score(
    y_test,
    xgboost_test_predictions,
    zero_division=0
)

xgboost_accuracy_gap = (
    xgboost_train_accuracy - xgboost_test_accuracy
)

print("=" * 60)
print("XGBOOST PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {xgboost_train_accuracy:.4f}")
print(f"Testing accuracy:  {xgboost_test_accuracy:.4f}")
print(f"Accuracy gap:      {xgboost_accuracy_gap:.4f}")
print(f"Precision:         {xgboost_precision:.4f}")
print(f"Recall:            {xgboost_recall:.4f}")
print(f"F1 score:          {xgboost_f1:.4f}")
print(f"Training time:     {xgboost_training_time:.6f} seconds")
print("=" * 60)

In [ ]:
# ==========================================================
# E5 - Store XGBoost Results
# ==========================================================

model_results.append({
    "Model": "XGBoost",
    "Training Accuracy": xgboost_train_accuracy,
    "Testing Accuracy": xgboost_test_accuracy,
    "Accuracy Gap": xgboost_accuracy_gap,
    "Precision": xgboost_precision,
    "Recall": xgboost_recall,
    "F1 Score": xgboost_f1,
    "Training Time": xgboost_training_time,
    "OOB Score": np.nan
})

print("XGBoost results stored successfully.")

In [ ]:
# ==========================================================
# E6 - XGBoost Confusion Matrix
# ==========================================================

xgboost_confusion_matrix = confusion_matrix(
    y_test,
    xgboost_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=xgboost_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("XGBoost Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E6 - XGBoost Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        xgboost_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## E7. LightGBM Classifier

LightGBM is a gradient-boosting framework designed for efficient training and strong predictive performance.

Unlike traditional level-wise tree construction, LightGBM commonly grows trees leaf-wise. This approach can reduce loss quickly, although model complexity must be controlled to avoid overfitting.

---

In [ ]:
# ==========================================================
# E7 - Train and Evaluate LightGBM
# ==========================================================

lightgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.10,
    max_depth=3,
    num_leaves=7,
    subsample=0.80,
    colsample_bytree=0.80,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

start_time = time.perf_counter()

lightgbm_model.fit(X_train, y_train)

lightgbm_training_time = time.perf_counter() - start_time

lightgbm_train_predictions = lightgbm_model.predict(X_train)
lightgbm_test_predictions = lightgbm_model.predict(X_test)

lightgbm_train_accuracy = accuracy_score(
    y_train,
    lightgbm_train_predictions
)

lightgbm_test_accuracy = accuracy_score(
    y_test,
    lightgbm_test_predictions
)

lightgbm_precision = precision_score(
    y_test,
    lightgbm_test_predictions,
    zero_division=0
)

lightgbm_recall = recall_score(
    y_test,
    lightgbm_test_predictions,
    zero_division=0
)

lightgbm_f1 = f1_score(
    y_test,
    lightgbm_test_predictions,
    zero_division=0
)

lightgbm_accuracy_gap = (
    lightgbm_train_accuracy - lightgbm_test_accuracy
)

print("=" * 60)
print("LIGHTGBM PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {lightgbm_train_accuracy:.4f}")
print(f"Testing accuracy:  {lightgbm_test_accuracy:.4f}")
print(f"Accuracy gap:      {lightgbm_accuracy_gap:.4f}")
print(f"Precision:         {lightgbm_precision:.4f}")
print(f"Recall:            {lightgbm_recall:.4f}")
print(f"F1 score:          {lightgbm_f1:.4f}")
print(f"Training time:     {lightgbm_training_time:.6f} seconds")
print("=" * 60)

In [ ]:
# ==========================================================
# E7 - Store LightGBM Results
# ==========================================================

model_results.append({
    "Model": "LightGBM",
    "Training Accuracy": lightgbm_train_accuracy,
    "Testing Accuracy": lightgbm_test_accuracy,
    "Accuracy Gap": lightgbm_accuracy_gap,
    "Precision": lightgbm_precision,
    "Recall": lightgbm_recall,
    "F1 Score": lightgbm_f1,
    "Training Time": lightgbm_training_time,
    "OOB Score": np.nan
})

print("LightGBM results stored successfully.")

In [ ]:
# ==========================================================
# E8 - LightGBM Confusion Matrix
# ==========================================================

lightgbm_confusion_matrix = confusion_matrix(
    y_test,
    lightgbm_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=lightgbm_confusion_matrix,
    display_labels=["No Heart Disease", "Heart Disease"]
)

display.plot(values_format="d")

plt.title("LightGBM Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E8 - LightGBM Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        lightgbm_test_predictions,
        target_names=["No Heart Disease", "Heart Disease"],
        digits=4,
        zero_division=0
    )
)

## E9. Boosting Model Comparison

The four boosting models are compared using the same evaluation measurements.

Testing accuracy measures overall correctness, while precision, recall, and F1 score provide a more detailed view of classification performance. The training-testing accuracy gap helps indicate possible overfitting, and training time shows the computational cost of each method.

---

In [ ]:
# ==========================================================
# E9 - Boosting Model Comparison Table
# ==========================================================

boosting_models = [
    "AdaBoost",
    "Gradient Boosting",
    "XGBoost",
    "LightGBM"
]

boosting_comparison = pd.DataFrame(model_results)

boosting_comparison = boosting_comparison[
    boosting_comparison["Model"].isin(boosting_models)
].copy()

boosting_comparison = boosting_comparison.sort_values(
    by="Testing Accuracy",
    ascending=False
).reset_index(drop=True)

boosting_comparison.insert(
    0,
    "Rank",
    range(1, len(boosting_comparison) + 1)
)

boosting_comparison.round(4)

In [ ]:
# ==========================================================
# E10 - Boosting Testing Accuracy Comparison
# ==========================================================

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=boosting_comparison,
    x="Model",
    y="Testing Accuracy"
)

plt.title("Boosting Model Testing Accuracy")
plt.xlabel("Model")
plt.ylabel("Testing Accuracy")
plt.ylim(0, 1.05)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E11 - Boosting Training versus Testing Accuracy
# ==========================================================

boosting_accuracy_comparison = boosting_comparison[
    [
        "Model",
        "Training Accuracy",
        "Testing Accuracy"
    ]
].set_index("Model")

boosting_accuracy_comparison.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("Boosting Model Training and Testing Accuracy")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# E12 - Boosting Training Time Comparison
# ==========================================================

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=boosting_comparison,
    x="Model",
    y="Training Time"
)

plt.title("Boosting Model Training Time")
plt.xlabel("Model")
plt.ylabel("Training Time in Seconds")

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3)

plt.tight_layout()
plt.show()

## E13. Boosting Model Summary

AdaBoost, Gradient Boosting, XGBoost, and LightGBM were trained and evaluated using the same stratified training and testing datasets.

AdaBoost adjusts observation weights to focus on previously misclassified records. Gradient Boosting reduces prediction errors sequentially. XGBoost adds optimization and regularization features, while LightGBM is designed for computational efficiency and fast tree construction.

The models should be compared using more than testing accuracy alone. In this medical classification problem, recall is especially important because a false negative represents a patient with heart disease who was incorrectly classified as not having the condition.

The most suitable boosting model should therefore provide strong testing accuracy, recall, and F1 performance without showing an excessive training-testing accuracy gap.

---

The next section combines all seven models into a single overall comparison.

<a id="part-f"></a>


# Part F – Overall Model Comparison

---

This section compares the performance of all seven machine learning models evaluated in this project:

- Decision Tree
- Bagging
- Random Forest
- AdaBoost
- Gradient Boosting
- XGBoost
- LightGBM

Rather than selecting a model based on a single metric, multiple performance measurements are considered, including:

- Training Accuracy
- Testing Accuracy
- Accuracy Gap
- Precision
- Recall
- F1 Score
- Training Time
- Out-of-Bag Score (when available)

These comparisons provide a comprehensive evaluation of predictive performance, computational efficiency, and generalization ability.

---

## F1. Overall Model Performance

The results collected throughout the notebook are combined into a single table. Models are ranked according to testing accuracy because this measurement best reflects performance on previously unseen observations.

---

In [ ]:
# ==========================================================
# F1 - Overall Model Comparison Table
# ==========================================================

overall_results = pd.DataFrame(model_results)

overall_results = overall_results.sort_values(
    by="Testing Accuracy",
    ascending=False
).reset_index(drop=True)

overall_results.insert(
    0,
    "Rank",
    range(1, len(overall_results) + 1)
)

overall_results.round(4)

## F2. Best Performing Model

The following code identifies the model that achieved the highest testing accuracy.

---

In [ ]:
# ==========================================================
# F2 - Best Performing Model
# ==========================================================

best_model = overall_results.iloc[0]

print("=" * 60)
print("BEST PERFORMING MODEL")
print("=" * 60)

print(f"Model:             {best_model['Model']}")
print(f"Testing Accuracy:  {best_model['Testing Accuracy']:.4f}")
print(f"Precision:         {best_model['Precision']:.4f}")
print(f"Recall:            {best_model['Recall']:.4f}")
print(f"F1 Score:          {best_model['F1 Score']:.4f}")
print(f"Training Time:     {best_model['Training Time']:.6f} sec")

print("=" * 60)

## F3. Testing Accuracy Comparison

Testing accuracy is the primary measure used to compare overall classification performance because it evaluates predictions on unseen data.

---

In [ ]:
# ==========================================================
# F3 - Testing Accuracy Comparison
# ==========================================================

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=overall_results,
    x="Model",
    y="Testing Accuracy"
)

plt.title("Testing Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Testing Accuracy")
plt.xticks(rotation=20)
plt.ylim(0,1.05)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")

plt.tight_layout()
plt.show()

## F4. Precision, Recall, and F1 Score Comparison

Although testing accuracy provides an overall measure of performance, precision, recall, and F1 score provide additional insight into each classifier's strengths.

Recall is especially important for this medical classification problem because it measures the model's ability to correctly identify patients with heart disease.

---

In [ ]:
# ==========================================================
# F4 - Precision, Recall and F1 Comparison
# ==========================================================

metric_comparison = overall_results[
    [
        "Model",
        "Precision",
        "Recall",
        "F1 Score"
    ]
].set_index("Model")

metric_comparison.plot(
    kind="bar",
    figsize=(12,6)
)

plt.title("Precision, Recall and F1 Score")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.ylim(0,1.05)

plt.tight_layout()
plt.show()

## F5. Training Accuracy versus Testing Accuracy

Comparing training and testing accuracy provides insight into model generalization.

A large difference between these values may indicate overfitting.

---

In [ ]:
# ==========================================================
# F5 - Training vs Testing Accuracy
# ==========================================================

accuracy_comparison = overall_results[
    [
        "Model",
        "Training Accuracy",
        "Testing Accuracy"
    ]
].set_index("Model")

accuracy_comparison.plot(
    kind="bar",
    figsize=(12,6)
)

plt.title("Training vs Testing Accuracy")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.ylim(0,1.05)

plt.tight_layout()
plt.show()

## F6. Accuracy Gap

The accuracy gap is calculated as:

Training Accuracy − Testing Accuracy

Smaller values generally indicate better generalization.

---

In [ ]:
# ==========================================================
# F6 - Accuracy Gap
# ==========================================================

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=overall_results,
    x="Model",
    y="Accuracy Gap"
)

plt.title("Training - Testing Accuracy Gap")
plt.xlabel("Model")
plt.ylabel("Accuracy Gap")
plt.xticks(rotation=20)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")

plt.tight_layout()
plt.show()

## F7. Training Time Comparison

Training time measures the computational cost of each model.

A model with only a slight improvement in predictive performance may not justify substantially longer training times.

---

In [ ]:
# ==========================================================
# F7 - Training Time Comparison
# ==========================================================

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=overall_results,
    x="Model",
    y="Training Time"
)

plt.title("Training Time Comparison")
plt.xlabel("Model")
plt.ylabel("Seconds")
plt.xticks(rotation=20)

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f")

plt.tight_layout()
plt.show()

## F8. Out-of-Bag Score Comparison

Out-of-Bag (OOB) scoring is available only for Bagging and Random Forest because these methods use bootstrap sampling.

The OOB score provides an internal estimate of model performance without requiring the testing dataset.

---

In [ ]:
# ==========================================================
# F8 - OOB Score Comparison
# ==========================================================

oob_models = overall_results[
    overall_results["OOB Score"].notna()
]

plt.figure(figsize=(7,5))

ax = sns.barplot(
    data=oob_models,
    x="Model",
    y="OOB Score"
)

plt.title("Out-of-Bag Score Comparison")
plt.ylim(0,1.05)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")

plt.tight_layout()
plt.show()

## F9. Overall Performance Rankings

The table below summarizes the overall rankings for all evaluated models based on testing accuracy.

Although ranking is useful, model selection should also consider recall, F1 score, overfitting, and computational efficiency.

---

In [ ]:
# ==========================================================
# F9 - Overall Rankings
# ==========================================================

ranking = overall_results[
    [
        "Rank",
        "Model",
        "Testing Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Training Time"
    ]
]

ranking.round(4)

## F10. Overall Comparison Summary

The seven machine learning models demonstrated different strengths in predictive performance, computational efficiency, and generalization.

The baseline Decision Tree established a useful point of comparison but may have exhibited greater overfitting than the ensemble models. Bagging and Random Forest improved stability by combining multiple trees, while the boosting algorithms focused on correcting previous prediction errors through sequential learning.

Among the boosting methods, XGBoost and LightGBM are designed to optimize predictive performance and computational efficiency through advanced gradient boosting techniques. The final model selection should balance testing accuracy, recall, F1 score, training time, and model complexity.

The next section examines feature importance to determine which patient characteristics contributed most to the predictions generated by the tree-based ensemble models.

---

<a id="part-g"></a>


# Part G – Feature Importance Analysis

---

Feature importance estimates how strongly each predictor contributes to a model's decisions.

This section compares feature importance from:

- Random Forest
- Gradient Boosting
- XGBoost
- LightGBM

The analysis identifies the patient characteristics that contributed most strongly to heart-disease classification. Because each algorithm calculates importance differently, the exact values should not be interpreted as directly equivalent. However, features that consistently rank highly across several models may be especially influential.

Feature importance indicates predictive contribution, not medical causation.

---

## G1. Random Forest Feature Importance

Random Forest calculates feature importance according to the total reduction in node impurity produced by each feature across all trees.

A larger value indicates that the feature contributed more frequently or more strongly to the model's decision rules.

---

In [ ]:
# ==========================================================
# G1 - Random Forest Feature Importance
# ==========================================================

random_forest_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

random_forest_importance.index = (
    random_forest_importance.index + 1
)

random_forest_importance

In [ ]:
# ==========================================================
# G1 - Random Forest Feature Importance Visualization
# ==========================================================

random_forest_plot = random_forest_importance.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    random_forest_plot["Feature"],
    random_forest_plot["Importance"]
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## G2. Gradient Boosting Feature Importance

Gradient Boosting calculates importance according to how much each feature reduces prediction error across the sequentially constructed trees.

Features with larger values made greater contributions to the ensemble's decision process.

---

In [ ]:
# ==========================================================
# G2 - Gradient Boosting Feature Importance
# ==========================================================

gradient_boosting_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": gradient_boosting_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

gradient_boosting_importance.index = (
    gradient_boosting_importance.index + 1
)

gradient_boosting_importance

In [ ]:
# ==========================================================
# G2 - Gradient Boosting Feature Importance Visualization
# ==========================================================

gradient_boosting_plot = gradient_boosting_importance.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    gradient_boosting_plot["Feature"],
    gradient_boosting_plot["Importance"]
)

plt.title("Gradient Boosting Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## G3. XGBoost Feature Importance

XGBoost provides feature-importance values based on the contribution of each feature to the boosted decision trees.

The following analysis uses the model's normalized importance values to rank the predictors.

---

In [ ]:
# ==========================================================
# G3 - XGBoost Feature Importance
# ==========================================================

xgboost_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgboost_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

xgboost_importance.index = (
    xgboost_importance.index + 1
)

xgboost_importance

In [ ]:
# ==========================================================
# G3 - XGBoost Feature Importance Visualization
# ==========================================================

xgboost_plot = xgboost_importance.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    xgboost_plot["Feature"],
    xgboost_plot["Importance"]
)

plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## G4. LightGBM Feature Importance

LightGBM reports how frequently each feature is used in the model's tree splits by default.

Because these values are raw split counts rather than normalized proportions, they are normalized below so that they can be compared more clearly with the other models.

---

In [ ]:
# ==========================================================
# G4 - LightGBM Feature Importance
# ==========================================================

lightgbm_raw_importance = (
    lightgbm_model.feature_importances_.astype(float)
)

lightgbm_normalized_importance = (
    lightgbm_raw_importance
    / lightgbm_raw_importance.sum()
)

lightgbm_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": lightgbm_normalized_importance
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

lightgbm_importance.index = (
    lightgbm_importance.index + 1
)

lightgbm_importance

In [ ]:
# ==========================================================
# G4 - LightGBM Feature Importance Visualization
# ==========================================================

lightgbm_plot = lightgbm_importance.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    lightgbm_plot["Feature"],
    lightgbm_plot["Importance"]
)

plt.title("LightGBM Feature Importance")
plt.xlabel("Normalized Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## G5. Combined Feature-Importance Comparison

The importance values from Random Forest, Gradient Boosting, XGBoost, and LightGBM are combined into one table.

Each column represents the normalized importance assigned by one model. The average importance provides a general indication of which features ranked highly across the ensemble methods.

Because the models calculate importance differently, the average should be viewed as a comparative summary rather than an absolute measurement.

---

In [ ]:
# ==========================================================
# G5 - Combined Feature Importance Table
# ==========================================================

feature_importance_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Random Forest": random_forest_model.feature_importances_,
    "Gradient Boosting": gradient_boosting_model.feature_importances_,
    "XGBoost": xgboost_model.feature_importances_,
    "LightGBM": lightgbm_normalized_importance
})

feature_importance_comparison["Average Importance"] = (
    feature_importance_comparison[
        [
            "Random Forest",
            "Gradient Boosting",
            "XGBoost",
            "LightGBM"
        ]
    ].mean(axis=1)
)

feature_importance_comparison = (
    feature_importance_comparison
    .sort_values(
        by="Average Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_comparison.index = (
    feature_importance_comparison.index + 1
)

feature_importance_comparison.round(4)

## G6. Feature Importance Across Models

The following grouped bar chart compares the importance assigned to each feature by the four ensemble models.

Differences between the bars demonstrate that models may rely on the same variables to different degrees. Features with consistently large bars across several models are likely to be influential throughout the analysis.

---

In [ ]:
# ==========================================================
# G6 - Combined Feature Importance Visualization
# ==========================================================

importance_plot_data = (
    feature_importance_comparison
    .set_index("Feature")
    [
        [
            "Random Forest",
            "Gradient Boosting",
            "XGBoost",
            "LightGBM"
        ]
    ]
)

importance_plot_data.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Feature Importance Across Ensemble Models")
plt.xlabel("Feature")
plt.ylabel("Normalized Importance")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.show()

## G7. Average Feature Importance Ranking

The average importance ranking summarizes the contribution of each feature across the four ensemble models.

This comparison helps identify variables that were consistently influential rather than appearing important in only one model.

---

In [ ]:
# ==========================================================
# G7 - Average Feature Importance
# ==========================================================

average_importance = (
    feature_importance_comparison
    .sort_values(
        by="Average Importance",
        ascending=True
    )
)

plt.figure(figsize=(10, 7))

plt.barh(
    average_importance["Feature"],
    average_importance["Average Importance"]
)

plt.title("Average Feature Importance Across Models")
plt.xlabel("Average Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## G8. Top Five Features by Model

The following table displays the five highest-ranked features for each model.

This makes it easier to determine whether the ensemble methods agree about the most influential predictors.

---

In [ ]:
# ==========================================================
# G8 - Top Five Features by Model
# ==========================================================

top_feature_summary = pd.DataFrame({
    "Rank": range(1, 6),
    "Random Forest": (
        random_forest_importance["Feature"]
        .head(5)
        .tolist()
    ),
    "Gradient Boosting": (
        gradient_boosting_importance["Feature"]
        .head(5)
        .tolist()
    ),
    "XGBoost": (
        xgboost_importance["Feature"]
        .head(5)
        .tolist()
    ),
    "LightGBM": (
        lightgbm_importance["Feature"]
        .head(5)
        .tolist()
    )
})

top_feature_summary

## G9. Top Overall Features

The following code identifies the five features with the highest average importance across the four ensemble models.

---

In [ ]:
# ==========================================================
# G9 - Top Overall Features
# ==========================================================

top_overall_features = (
    feature_importance_comparison
    .head(5)
    [
        [
            "Feature",
            "Average Importance"
        ]
    ]
)

print("=" * 60)
print("TOP FIVE FEATURES ACROSS ENSEMBLE MODELS")
print("=" * 60)

for rank, (_, row) in enumerate(
    top_overall_features.iterrows(),
    start=1
):
    print(
        f"{rank}. {row['Feature']:<12} "
        f"{row['Average Importance']:.4f}"
    )

print("=" * 60)

## G10. Random Forest versus LightGBM Feature Importance

The assignment specifically calls for comparing feature importance from Random Forest and LightGBM.

Random Forest importance reflects accumulated impurity reduction across its independently trained trees. LightGBM importance reflects how frequently features were selected for splits during sequential boosting.

The following table displays the rankings and importance values produced by both models.

---

In [ ]:
# ==========================================================
# G10 - Random Forest versus LightGBM
# ==========================================================

rf_lgbm_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Random Forest Importance":
        random_forest_model.feature_importances_,
    "LightGBM Importance":
        lightgbm_normalized_importance
})

rf_lgbm_comparison["Random Forest Rank"] = (
    rf_lgbm_comparison[
        "Random Forest Importance"
    ].rank(
        ascending=False,
        method="min"
    ).astype(int)
)

rf_lgbm_comparison["LightGBM Rank"] = (
    rf_lgbm_comparison[
        "LightGBM Importance"
    ].rank(
        ascending=False,
        method="min"
    ).astype(int)
)

rf_lgbm_comparison["Rank Difference"] = (
    rf_lgbm_comparison["Random Forest Rank"]
    - rf_lgbm_comparison["LightGBM Rank"]
).abs()

rf_lgbm_comparison = (
    rf_lgbm_comparison
    .sort_values(
        by="Random Forest Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

rf_lgbm_comparison.index = (
    rf_lgbm_comparison.index + 1
)

rf_lgbm_comparison.round(4)

In [ ]:
# ==========================================================
# G10 - Random Forest versus LightGBM Visualization
# ==========================================================

rf_lgbm_plot = (
    rf_lgbm_comparison
    .set_index("Feature")
    [
        [
            "Random Forest Importance",
            "LightGBM Importance"
        ]
    ]
)

rf_lgbm_plot.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Random Forest versus LightGBM Feature Importance")
plt.xlabel("Feature")
plt.ylabel("Normalized Importance")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.show()

## G11. Feature-Importance Interpretation

The feature-importance analysis shows that the models did not assign equal importance to every patient characteristic. Across the ensemble methods, the most influential variables included `[Feature 1]`, `[Feature 2]`, `[Feature 3]`, `[Feature 4]`, and `[Feature 5]`.

Random Forest and LightGBM showed both agreement and disagreement in their rankings. Features ranked highly by both models appear to provide consistently useful information for classifying heart disease. Differences are expected because Random Forest creates many independent bootstrap-trained trees, while LightGBM constructs trees sequentially to correct previous errors.

The importance values do not show whether a feature increases or decreases heart-disease risk. They only estimate how useful the feature was when making predictions. They also do not establish a causal medical relationship.

---

## G12. Feature-Importance Summary

Feature importance was evaluated using Random Forest, Gradient Boosting, XGBoost, and LightGBM.

The analysis identified several variables that consistently contributed to the model predictions. Although the exact rankings differed among the algorithms, features with high average importance appeared influential across multiple ensemble-learning approaches.

The Random Forest and LightGBM comparison demonstrated that model architecture affects how predictor importance is assigned. Random Forest measures contributions across independently constructed trees, while LightGBM emphasizes variables selected during sequential error correction.

Feature importance improves model interpretability, but it should not be treated as proof that a predictor causes heart disease. The results describe relationships learned from this dataset and should be interpreted alongside clinical knowledge and additional validation.

---

The next section applies hyperparameter tuning and cross-validation to improve model selection and evaluate performance more reliably.

<a id="part-h"></a>


# Part H – Hyperparameter Tuning and Cross-Validation

---

The earlier model comparisons used a single stratified training and testing split. Although this provides a consistent evaluation, performance can vary depending on which observations are assigned to each subset.

Cross-validation provides a more reliable estimate by evaluating a model across multiple data splits. Hyperparameter tuning searches for model settings that improve predictive performance while reducing overfitting.

This section focuses on the Random Forest classifier because it:

- Provides strong generalization potential
- Supports Out-of-Bag evaluation
- Produces interpretable feature-importance values
- Includes several adjustable hyperparameters
- Is specifically identified in the assignment's optional bonus section

The tuning process includes:

- Establishing an untuned cross-validation baseline
- Defining a Random Forest parameter grid
- Running GridSearchCV
- Reviewing the best parameters
- Evaluating the tuned model on the testing set
- Comparing tuned and untuned performance
- Performing final cross-validation on the tuned model

---

## H1. Configure Stratified Cross-Validation

A five-fold stratified cross-validation procedure is used.

The training data is divided into five subsets. During each iteration, four subsets are used for training and one is used for validation. This process repeats until every subset has served as the validation fold once.

Stratification preserves approximately the same target-class proportions within each fold.

---

In [ ]:
# ==========================================================
# H1 - Configure Stratified Cross-Validation
# ==========================================================

stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("STRATIFIED CROSS-VALIDATION CONFIGURED")
print("=" * 60)
print(f"Number of folds: {stratified_cv.n_splits}")
print("Shuffling:       Enabled")
print(f"Random state:    {RANDOM_STATE}")
print("=" * 60)

## H2. Untuned Random Forest Cross-Validation

Before tuning the Random Forest, five-fold cross-validation is performed using the original model settings.

This establishes a baseline for determining whether hyperparameter tuning improves average validation accuracy.

---

In [ ]:
# ==========================================================
# H2 - Untuned Random Forest Cross-Validation
# ==========================================================

untuned_cv_scores = cross_val_score(
    random_forest_model,
    X_train,
    y_train,
    cv=stratified_cv,
    scoring="accuracy",
    n_jobs=-1
)

untuned_cv_mean = untuned_cv_scores.mean()
untuned_cv_std = untuned_cv_scores.std()

print("=" * 60)
print("UNTUNED RANDOM FOREST CROSS-VALIDATION")
print("=" * 60)

for fold_number, score in enumerate(
    untuned_cv_scores,
    start=1
):
    print(f"Fold {fold_number}: {score:.4f}")

print("-" * 60)
print(f"Mean accuracy:     {untuned_cv_mean:.4f}")
print(f"Standard deviation:{untuned_cv_std:.4f}")
print("=" * 60)

## H3. Define the Random Forest Parameter Grid

The parameter grid defines the Random Forest configurations that GridSearchCV will evaluate.

The tuned hyperparameters are:

- `n_estimators`: Number of trees in the forest
- `max_depth`: Maximum depth of each tree
- `min_samples_split`: Minimum observations required to split an internal node
- `min_samples_leaf`: Minimum observations required in a leaf
- `max_features`: Number of features considered at each split
- `criterion`: Function used to measure split quality

Limiting tree depth or increasing minimum sample requirements may reduce overfitting.

In [ ]:
# ==========================================================
# H3 - Define Random Forest Parameter Grid
# ==========================================================

random_forest_parameter_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "criterion": ["gini", "entropy"]
}

total_parameter_combinations = (
    len(random_forest_parameter_grid["n_estimators"])
    * len(random_forest_parameter_grid["max_depth"])
    * len(random_forest_parameter_grid["min_samples_split"])
    * len(random_forest_parameter_grid["min_samples_leaf"])
    * len(random_forest_parameter_grid["max_features"])
    * len(random_forest_parameter_grid["criterion"])
)

total_model_fits = (
    total_parameter_combinations
    * stratified_cv.n_splits
)

print("=" * 60)
print("RANDOM FOREST PARAMETER GRID")
print("=" * 60)
print(f"Parameter combinations: {total_parameter_combinations}")
print(f"Cross-validation folds: {stratified_cv.n_splits}")
print(f"Estimated model fits:   {total_model_fits}")
print("=" * 60)

## H4. Run Random Forest Grid Search

GridSearchCV trains and evaluates every parameter combination using the five-fold stratified cross-validation procedure.

The parameter combination with the highest mean validation accuracy is selected as the tuned Random Forest model.

---

In [ ]:
# ==========================================================
# H4 - Run Random Forest GridSearchCV
# ==========================================================

random_forest_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        bootstrap=True,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_grid=random_forest_parameter_grid,
    scoring="accuracy",
    cv=stratified_cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search_start_time = time.perf_counter()

random_forest_grid_search.fit(
    X_train,
    y_train
)

grid_search_training_time = (
    time.perf_counter() - grid_search_start_time
)

print()
print("=" * 60)
print("RANDOM FOREST GRID SEARCH COMPLETE")
print("=" * 60)
print(
    f"Best cross-validation accuracy: "
    f"{random_forest_grid_search.best_score_:.4f}"
)
print(
    f"Grid-search time: "
    f"{grid_search_training_time:.4f} seconds"
)
print("=" * 60)

## H5. Best Random Forest Hyperparameters

The following output displays the parameter combination that achieved the highest average cross-validation accuracy.

---

In [ ]:
# ==========================================================
# H5 - Display Best Random Forest Parameters
# ==========================================================

best_random_forest_parameters = (
    random_forest_grid_search.best_params_
)

print("=" * 60)
print("BEST RANDOM FOREST PARAMETERS")
print("=" * 60)

for parameter, value in best_random_forest_parameters.items():
    print(f"{parameter:<20}: {value}")

print("=" * 60)

## H6. Top Grid-Search Results

The following table displays the ten highest-performing parameter combinations.

The mean training score and mean validation score help reveal whether a configuration fits the training folds substantially better than the validation folds.

---

In [ ]:
# ==========================================================
# H6 - Top Grid-Search Results
# ==========================================================

grid_search_results = pd.DataFrame(
    random_forest_grid_search.cv_results_
)

top_grid_results = grid_search_results[
    [
        "rank_test_score",
        "mean_train_score",
        "std_train_score",
        "mean_test_score",
        "std_test_score",
        "mean_fit_time",
        "params"
    ]
].sort_values(
    by="rank_test_score"
).head(10)

top_grid_results = top_grid_results.rename(
    columns={
        "rank_test_score": "Rank",
        "mean_train_score": "Mean Training Accuracy",
        "std_train_score": "Training Std.",
        "mean_test_score": "Mean Validation Accuracy",
        "std_test_score": "Validation Std.",
        "mean_fit_time": "Mean Fit Time",
        "params": "Parameters"
    }
)

top_grid_results.reset_index(
    drop=True
).round(4)

## H7. Retrieve the Tuned Random Forest

GridSearchCV automatically retrains the best-performing parameter configuration using the complete training dataset.

The resulting estimator is stored as `tuned_random_forest_model`.

In [ ]:
# ==========================================================
# H7 - Retrieve Tuned Random Forest
# ==========================================================

tuned_random_forest_model = (
    random_forest_grid_search.best_estimator_
)

print("=" * 60)
print("TUNED RANDOM FOREST RETRIEVED")
print("=" * 60)
print(
    f"Number of estimators: "
    f"{tuned_random_forest_model.n_estimators}"
)
print(
    f"Maximum depth:       "
    f"{tuned_random_forest_model.max_depth}"
)
print(
    f"Minimum split size:  "
    f"{tuned_random_forest_model.min_samples_split}"
)
print(
    f"Minimum leaf size:   "
    f"{tuned_random_forest_model.min_samples_leaf}"
)
print(
    f"Maximum features:    "
    f"{tuned_random_forest_model.max_features}"
)
print(
    f"Split criterion:     "
    f"{tuned_random_forest_model.criterion}"
)
print("=" * 60)

## H8. Evaluate the Tuned Random Forest

The tuned model is evaluated on the original training and testing datasets.

Using the same testing set allows direct comparison with the untuned Random Forest and the other models evaluated earlier.

In [ ]:
# ==========================================================
# H8 - Evaluate Tuned Random Forest
# ==========================================================

tuned_rf_train_predictions = (
    tuned_random_forest_model.predict(X_train)
)

tuned_rf_test_predictions = (
    tuned_random_forest_model.predict(X_test)
)

tuned_rf_train_accuracy = accuracy_score(
    y_train,
    tuned_rf_train_predictions
)

tuned_rf_test_accuracy = accuracy_score(
    y_test,
    tuned_rf_test_predictions
)

tuned_rf_precision = precision_score(
    y_test,
    tuned_rf_test_predictions,
    zero_division=0
)

tuned_rf_recall = recall_score(
    y_test,
    tuned_rf_test_predictions,
    zero_division=0
)

tuned_rf_f1 = f1_score(
    y_test,
    tuned_rf_test_predictions,
    zero_division=0
)

tuned_rf_accuracy_gap = (
    tuned_rf_train_accuracy
    - tuned_rf_test_accuracy
)

tuned_rf_oob_score = (
    tuned_random_forest_model.oob_score_
)

print("=" * 60)
print("TUNED RANDOM FOREST PERFORMANCE")
print("=" * 60)
print(f"Training accuracy: {tuned_rf_train_accuracy:.4f}")
print(f"Testing accuracy:  {tuned_rf_test_accuracy:.4f}")
print(f"Accuracy gap:      {tuned_rf_accuracy_gap:.4f}")
print(f"Precision:         {tuned_rf_precision:.4f}")
print(f"Recall:            {tuned_rf_recall:.4f}")
print(f"F1 score:          {tuned_rf_f1:.4f}")
print(f"Out-of-Bag score:  {tuned_rf_oob_score:.4f}")
print("=" * 60)

## H9. Tuned Random Forest Confusion Matrix

The confusion matrix displays the tuned model's correct classifications and errors on the testing dataset.

Particular attention should be given to false negatives because they represent patients with heart disease who were incorrectly classified as not having the condition.

---

In [ ]:
# ==========================================================
# H9 - Tuned Random Forest Confusion Matrix
# ==========================================================

tuned_rf_confusion_matrix = confusion_matrix(
    y_test,
    tuned_rf_test_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=tuned_rf_confusion_matrix,
    display_labels=[
        "No Heart Disease",
        "Heart Disease"
    ]
)

display.plot(values_format="d")

plt.title("Tuned Random Forest Confusion Matrix")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# H10 - Tuned Random Forest Classification Report
# ==========================================================

print(
    classification_report(
        y_test,
        tuned_rf_test_predictions,
        target_names=[
            "No Heart Disease",
            "Heart Disease"
        ],
        digits=4,
        zero_division=0
    )
)

## H11. Tuned Random Forest Cross-Validation

The tuned model is evaluated again using five-fold stratified cross-validation.

The mean accuracy provides a more stable estimate of expected performance than a single train-test split, while the standard deviation shows how much performance varied across the folds.

In [ ]:
# ==========================================================
# H11 - Tuned Random Forest Cross-Validation
# ==========================================================

tuned_cv_scores = cross_val_score(
    tuned_random_forest_model,
    X_train,
    y_train,
    cv=stratified_cv,
    scoring="accuracy",
    n_jobs=-1
)

tuned_cv_mean = tuned_cv_scores.mean()
tuned_cv_std = tuned_cv_scores.std()

print("=" * 60)
print("TUNED RANDOM FOREST CROSS-VALIDATION")
print("=" * 60)

for fold_number, score in enumerate(
    tuned_cv_scores,
    start=1
):
    print(f"Fold {fold_number}: {score:.4f}")

print("-" * 60)
print(f"Mean accuracy:     {tuned_cv_mean:.4f}")
print(f"Standard deviation:{tuned_cv_std:.4f}")
print("=" * 60)

## H12. Untuned versus Tuned Random Forest

The untuned and tuned Random Forest models are compared using testing accuracy, cross-validation accuracy, accuracy gap, precision, recall, F1 score, and Out-of-Bag score.

An effective tuning process should improve validation performance or reduce overfitting without substantially harming recall.

In [ ]:
# ==========================================================
# H12 - Untuned versus Tuned Random Forest
# ==========================================================

random_forest_tuning_comparison = pd.DataFrame({
    "Metric": [
        "Training Accuracy",
        "Testing Accuracy",
        "Accuracy Gap",
        "Precision",
        "Recall",
        "F1 Score",
        "OOB Score",
        "Mean CV Accuracy",
        "CV Standard Deviation"
    ],
    "Untuned Random Forest": [
        random_forest_train_accuracy,
        random_forest_test_accuracy,
        random_forest_accuracy_gap,
        random_forest_precision,
        random_forest_recall,
        random_forest_f1,
        random_forest_model.oob_score_,
        untuned_cv_mean,
        untuned_cv_std
    ],
    "Tuned Random Forest": [
        tuned_rf_train_accuracy,
        tuned_rf_test_accuracy,
        tuned_rf_accuracy_gap,
        tuned_rf_precision,
        tuned_rf_recall,
        tuned_rf_f1,
        tuned_rf_oob_score,
        tuned_cv_mean,
        tuned_cv_std
    ]
})

random_forest_tuning_comparison.round(4)

In [ ]:
# ==========================================================
# H13 - Untuned versus Tuned Performance Visualization
# ==========================================================

tuning_plot_data = (
    random_forest_tuning_comparison[
        random_forest_tuning_comparison["Metric"].isin(
            [
                "Testing Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "OOB Score",
                "Mean CV Accuracy"
            ]
        )
    ]
    .set_index("Metric")
)

tuning_plot_data.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.title("Untuned versus Tuned Random Forest")
plt.xlabel("Evaluation Metric")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# H14 - Cross-Validation Fold Comparison
# ==========================================================

cv_fold_comparison = pd.DataFrame({
    "Fold": range(1, stratified_cv.n_splits + 1),
    "Untuned Random Forest": untuned_cv_scores,
    "Tuned Random Forest": tuned_cv_scores
})

cv_fold_comparison

In [ ]:
# ==========================================================
# H14 - Cross-Validation Fold Visualization
# ==========================================================

cv_fold_plot = cv_fold_comparison.set_index("Fold")

cv_fold_plot.plot(
    kind="line",
    marker="o",
    figsize=(10, 6)
)

plt.title("Random Forest Cross-Validation Accuracy by Fold")
plt.xlabel("Fold")
plt.ylabel("Validation Accuracy")
plt.ylim(0, 1.05)
plt.xticks(cv_fold_comparison["Fold"])
plt.tight_layout()
plt.show()

## H15. Cross-Validation Improvement

The following calculation measures the difference between the tuned and untuned mean cross-validation accuracies.

A positive value indicates that tuning improved average validation performance, while a negative value indicates that the untuned model performed better during cross-validation.

In [ ]:
# ==========================================================
# H15 - Calculate Cross-Validation Improvement
# ==========================================================

cv_improvement = (
    tuned_cv_mean - untuned_cv_mean
)

test_accuracy_improvement = (
    tuned_rf_test_accuracy
    - random_forest_test_accuracy
)

gap_reduction = (
    random_forest_accuracy_gap
    - tuned_rf_accuracy_gap
)

print("=" * 60)
print("RANDOM FOREST TUNING IMPACT")
print("=" * 60)
print(
    f"Cross-validation improvement: "
    f"{cv_improvement:+.4f}"
)
print(
    f"Testing accuracy improvement: "
    f"{test_accuracy_improvement:+.4f}"
)
print(
    f"Accuracy-gap reduction:       "
    f"{gap_reduction:+.4f}"
)
print("=" * 60)

## H16. Add the Tuned Random Forest to the Model Results

The tuned Random Forest is added to the model-results collection so that it can be included in the final comparison.

The reported training time uses the complete GridSearchCV process because that represents the computational cost required to identify and train the tuned model.

In [ ]:
# ==========================================================
# H16 - Store Tuned Random Forest Results
# ==========================================================

model_results = [
    result for result in model_results
    if result["Model"] != "Tuned Random Forest"
]

model_results.append({
    "Model": "Tuned Random Forest",
    "Training Accuracy": tuned_rf_train_accuracy,
    "Testing Accuracy": tuned_rf_test_accuracy,
    "Accuracy Gap": tuned_rf_accuracy_gap,
    "Precision": tuned_rf_precision,
    "Recall": tuned_rf_recall,
    "F1 Score": tuned_rf_f1,
    "Training Time": grid_search_training_time,
    "OOB Score": tuned_rf_oob_score
})

print("Tuned Random Forest results stored successfully.")

In [ ]:
# ==========================================================
# H17 - Updated Overall Model Comparison
# ==========================================================

updated_overall_results = pd.DataFrame(
    model_results
)

updated_overall_results = (
    updated_overall_results
    .sort_values(
        by="Testing Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

updated_overall_results.insert(
    0,
    "Rank",
    range(
        1,
        len(updated_overall_results) + 1
    )
)

updated_overall_results.round(4)

## H18. Hyperparameter-Tuning Interpretation

Hyperparameter tuning improved the Random Forest's average cross-validation accuracy compared with the original configuration.

The tuned model's testing accuracy, recall, F1 score, Out-of-Bag score, and training-testing accuracy gap were also reviewed. Although the testing result remains important, the cross-validation average provides a more dependable estimate because it evaluates the model across multiple validation subsets.

The tuning process required considerably more computation than training one Random Forest. However, it provided a systematic method for selecting model complexity rather than relying on default settings.

## H19. Hyperparameter Tuning and Cross-Validation Summary

Five-fold stratified cross-validation was used to evaluate the Random Forest more reliably than a single train-test split. The untuned model first established a baseline cross-validation score.

GridSearchCV then evaluated combinations of tree count, maximum depth, split requirements, leaf requirements, feature selection, and split criterion. The best configuration was retrained using the full training dataset and evaluated on the reserved testing set.

The tuned and untuned Random Forest models were compared using testing accuracy, precision, recall, F1 score, Out-of-Bag score, cross-validation accuracy, and the training-testing accuracy gap.

Cross-validation is especially important for this project because the cleaned dataset contains only 302 unique observations. Evaluating multiple folds reduces dependence on one particular test split and provides a more stable estimate of expected model performance.

---

The next section presents the project reflection, final model recommendation, and conclusion.

<a id="part-i"></a>


# Part I – Reflection and Conclusion

---

The purpose of this project was to compare multiple ensemble learning algorithms for predicting heart disease using the Heart Disease dataset. Seven supervised machine learning models were developed, evaluated, and compared using identical training and testing datasets. Model performance was measured using testing accuracy, precision, recall, F1 score, training time, Out-of-Bag (OOB) score where applicable, feature importance analysis, and cross-validation.

The project demonstrated that different ensemble-learning methods exhibit different strengths. Although ensemble methods generally improve robustness over a single decision tree, no single model dominated every evaluation metric. Selecting an appropriate model therefore requires balancing predictive performance, generalization, interpretability, and computational cost.

---

## I1. Project Objectives Review

The primary objectives of this project were successfully completed.

- A real-world heart disease dataset was explored and cleaned.
- Duplicate observations were removed prior to model development.
- Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost, and LightGBM classifiers were implemented.
- Multiple evaluation metrics were used to compare model performance.
- Feature importance was analyzed across several ensemble models.
- Hyperparameter tuning and five-fold stratified cross-validation were performed to improve the Random Forest classifier.
- The final models were compared to determine the most appropriate classifier for this dataset.

The project demonstrates a complete supervised machine learning workflow, including data preparation, model development, evaluation, interpretation, and optimization.

---

## I2. Final Model Comparison

The Decision Tree classifier achieved the highest testing accuracy during the initial model evaluation. However, it also achieved perfect training accuracy, indicating substantial overfitting.

The tuned Random Forest classifier provided a better balance between predictive performance and generalization. Hyperparameter tuning increased both testing accuracy and average cross-validation accuracy while reducing the difference between training and testing performance. These improvements suggest that the tuned Random Forest is likely to perform more consistently on unseen data than the original untuned model.

Although XGBoost and LightGBM are frequently among the highest-performing ensemble methods, they did not outperform the tuned Random Forest on this relatively small dataset. This outcome illustrates that more sophisticated algorithms do not always provide superior results, particularly when the available training data are limited.

---

## I3. Lessons Learned

Several important machine learning concepts were reinforced throughout this project.

First, higher training accuracy does not necessarily indicate a better model. The Decision Tree perfectly memorized the training data but generalized less effectively than several ensemble approaches.

Second, evaluating multiple performance metrics is essential. Accuracy alone does not fully describe classifier performance, particularly in healthcare applications where false negatives may have serious consequences. Precision, recall, F1 score, and confusion matrices provide valuable additional insight.

Third, cross-validation offers a more reliable estimate of expected model performance than a single train-test split because it evaluates the model across multiple validation subsets.

Finally, hyperparameter tuning demonstrated that carefully selecting model parameters can improve predictive performance while simultaneously reducing overfitting.

---

## I4. Project Limitations

Several limitations should be considered when interpreting these results.

The cleaned dataset contained only 302 unique observations, limiting the amount of information available for training complex ensemble models. Performance measurements may therefore vary depending on the specific train-test split.

Additionally, the project focused on classification accuracy rather than clinical decision-making. Feature importance estimates identify variables that were useful for prediction but do not establish causal relationships between patient characteristics and heart disease.

Finally, only a subset of possible hyperparameter combinations was evaluated. Additional optimization techniques, such as randomized search or Bayesian optimization, could potentially produce further improvements.

---

## I5. Future Improvements

Several enhancements could further improve this project.

Future work could include evaluating larger and more diverse heart disease datasets to improve model generalization. Additional ensemble methods, such as CatBoost or Extremely Randomized Trees (Extra Trees), could also be compared.

Model explainability techniques such as SHAP (SHapley Additive exPlanations) or LIME could provide more detailed explanations of individual predictions. These approaches are increasingly important in healthcare applications where clinicians must understand why a model produced a particular recommendation.

Additional hyperparameter optimization strategies and repeated cross-validation could also provide more robust estimates of expected predictive performance.

---

## I6. Professional Reflection

This project provided valuable experience with the complete machine learning lifecycle. Beyond implementing classification algorithms, it required careful data preparation, systematic experimentation, model evaluation, feature interpretation, and performance optimization.

Working with multiple ensemble-learning techniques demonstrated that selecting an appropriate model involves balancing predictive accuracy, computational efficiency, interpretability, and the ability to generalize to unseen data. These considerations closely reflect real-world data science and machine learning practice, where the highest-performing model is not always the most appropriate solution.

The experience gained through this project strengthens practical skills in Python, Scikit-learn, XGBoost, LightGBM, exploratory data analysis, supervised learning, and model evaluation.

---

## I7. Final Conclusion

This project successfully demonstrated the application of ensemble learning techniques to a heart disease classification problem. Multiple supervised machine learning algorithms were implemented and evaluated using a consistent experimental methodology, allowing meaningful comparisons of predictive performance, computational efficiency, and generalization ability.

Although the Decision Tree classifier achieved the highest testing accuracy on the initial train-test split, it exhibited substantial overfitting. After hyperparameter tuning and cross-validation, the Random Forest classifier demonstrated improved generalization, higher average validation accuracy, increased testing accuracy, and a reduced training-testing accuracy gap. These results support the tuned Random Forest as the most balanced model for this dataset.

Overall, the project illustrates that successful machine learning requires more than maximizing accuracy. Careful data preparation, appropriate evaluation metrics, cross-validation, hyperparameter optimization, and thoughtful interpretation are all essential components of developing reliable predictive models. The workflow presented in this notebook provides a practical example of applying ensemble learning methods to a real-world healthcare classification problem.

---

<a id="part-j"></a>


In [ ]:
# Part J – Kaggle Notebook

---

This notebook was successfully developed and tested in both a local Jupyter Notebook environment and a Kaggle Notebook environment. The completed notebook satisfies the project requirements and is intended for publication on Kaggle before Blackboard submission.
Kaggle Notebook

Notebook URL:

https://www.kaggle.com/code/julianchristianiii/heart_disease_ensemble_learning
---